# Bengali OCR Qwen2-VL: Real-Time Metrics & Loss Dashboard
Run this notebook on a **free CPU runtime** (parallel to your GPU training) to visualize all training and validation metrics directly from your Google Drive checkpoints.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Load latest checkpoint logs and render all graphs
import os, glob, json
import matplotlib.pyplot as plt

# Search for checkpoints in Google Drive
checkpoints = sorted(glob.glob('/content/drive/MyDrive/bangla_ocr_checkpoints/checkpoint-*'),
                     key=lambda x: int(x.split('-')[-1]) if x.split('-')[-1].isdigit() else 0)

if not checkpoints:
    raise FileNotFoundError('No checkpoints found in /content/drive/MyDrive/bangla_ocr_checkpoints/')

latest_cp = checkpoints[-1]
state_file = os.path.join(latest_cp, 'trainer_state.json')
print(f'Loading metrics from latest checkpoint: {latest_cp}')

with open(state_file, 'r') as f:
    st = json.load(f)

logs = st['log_history']
train_entries = [l for l in logs if 'loss' in l]
eval_entries = [l for l in logs if 'eval_loss' in l]

train_steps = [l['step'] for l in train_entries]
train_losses = [l['loss'] for l in train_entries]
lrs = [l.get('learning_rate', 0) for l in train_entries]
grad_norms = [l.get('grad_norm', 0) for l in train_entries]

eval_steps = [l['step'] for l in eval_entries]
eval_losses = [l['eval_loss'] for l in eval_entries]

# Calculate EMA for smoothing
def calc_ema(values, alpha=0.08):
    ema = []
    for v in values:
        if not ema:
            ema.append(v)
        else:
            ema.append(alpha * v + (1 - alpha) * ema[-1])
    return ema

ema_loss = calc_ema(train_losses)
ema_grad = calc_ema(grad_norms, alpha=0.05)

# Plot 4-Panel Metrics Dashboard
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axs = plt.subplots(2, 2, figsize=(18, 12), dpi=150)

# 1. Full Loss (Log Scale)
ax1 = axs[0, 0]
ax1.plot(train_steps, train_losses, color='#3a86ff', alpha=0.3, label='Train Loss (raw)')
ax1.plot(train_steps, ema_loss, color='#1d3557', linewidth=2.2, label='Train Loss (Smoothed EMA)')
ax1.plot(eval_steps, eval_losses, 'o-', color='#e63946', linewidth=2.2, markersize=5, label='Validation Loss')
if 1000 <= max(train_steps):
    ax1.axvline(1000, color='#8338ec', linestyle='--', alpha=0.7, label='Step 1,000')
ax1.axvline(train_steps[-1], color='#06d6a0', linestyle='--', alpha=0.7, label=f'Current: Step {train_steps[-1]}')
ax1.set_yscale('log')
ax1.set_title(f'1. Overall Loss Trajectory (Steps 0 - {train_steps[-1]})', fontsize=13, fontweight='bold')
ax1.set_xlabel('Global Steps')
ax1.set_ylabel('Loss (Log Scale)')
ax1.legend(loc='upper right')
ax1.grid(True, which='both', linestyle='--', alpha=0.5)

# 2. Detailed Convergence (Linear Scale)
ax2 = axs[0, 1]
mask = [s >= 300 for s in train_steps]
z_steps = [s for s, m in zip(train_steps, mask) if m]
z_loss = [l for l, m in zip(train_losses, mask) if m]
z_ema = [e for e, m in zip(ema_loss, mask) if m]
z_eval_steps = [s for s in eval_steps if s >= 300]
z_eval_losses = [l for s, l in zip(eval_steps, eval_losses) if s >= 300]

ax2.plot(z_steps, z_loss, color='#457b9d', alpha=0.3, label='Train Loss (raw)')
ax2.plot(z_steps, z_ema, color='#1d3557', linewidth=2.2, label='Train Loss (Smoothed)')
ax2.plot(z_eval_steps, z_eval_losses, 's-', color='#e63946', linewidth=2.2, markersize=5, label='Validation Loss')
ax2.set_title('2. Detailed Convergence Zone (Linear Scale, Steps 300+)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Global Steps')
ax2.set_ylabel('Loss (Linear)')
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.5)

# 3. Learning Rate Schedule
ax3 = axs[1, 0]
ax3.plot(train_steps, lrs, color='#fb8500', linewidth=2.2, label='Learning Rate')
ax3.set_title('3. Cosine Learning Rate Schedule', fontsize=13, fontweight='bold')
ax3.set_xlabel('Global Steps')
ax3.set_ylabel('Learning Rate')
ax3.legend(loc='lower left')
ax3.grid(True, linestyle='--', alpha=0.5)

# 4. Gradient Norm (Stability)
ax4 = axs[1, 1]
ax4.plot(train_steps, grad_norms, color='#8ecae6', alpha=0.35, label='Grad Norm (raw)')
ax4.plot(train_steps, ema_grad, color='#023047', linewidth=2.0, label='Grad Norm (Smoothed)')
ax4.axhline(1.0, color='#d62828', linestyle=':', label='Baseline Target (~1.0)')
ax4.set_title('4. Gradient Norm & Optimization Stability', fontsize=13, fontweight='bold')
ax4.set_xlabel('Global Steps')
ax4.set_ylabel('Gradient Norm')
ax4.legend(loc='upper right')
ax4.grid(True, linestyle='--', alpha=0.5)

plt.suptitle(f'Qwen2-VL Bengali OCR: Metrics Dashboard (Current: Step {train_steps[-1]} / Loss: {train_losses[-1]:.4f})', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()